# Notebook 14 — BONUS: cross-journal / specialty / institutional **seeding**

**EXPLORATORY & NON-GATING.** Everything in this notebook is a *bonus* descriptive
layer. It **never** enters the Gate G5 confirmed-finding count, makes **no** causal
claim, and every figure is labelled `exploratory / non-gating`. The "seeding" signal
is purely temporal — within a research topic, *which entity tends to publish first*
(seeds) vs *which tends to follow* (lags). $0, CPU-only, **read-only** on existing
artifacts; no network / GPU / DeepSeek.

It imports the pure logic from `scifield.findings.seeding` (it does **not**
reimplement any statistic) and does all I/O here: `data/v1/archetypes.parquet` plus
the DuckDB enrichment tables `paper_institutions` + `institutions`.

Three sub-analyses, ≤3 figures:
1. **Cross-journal seeding** — per-journal lead/follow score + a directed,
   weight-thresholded seeding network → `bonus_seeding_network.png`.
2. **Per-specialty contrast** (orthopedic vs general_surgery) — F2 `arch_mean_cd5`
   archetype *mix* and seeding patterns by specialty → `bonus_specialty_contrast.png`.
3. **Institutional / geographic seeding** — top seeding institutions and countries,
   gated by an up-front metadata-availability guard → `bonus_geographic.png`.

## 1. Setup

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import duckdb
import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

# Repo-root sniff — notebook runs from notebooks/, code lives one dir up.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

DATA = repo_root / "data" / "v1"
FIGURES_DIR = repo_root / "docs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DPI = 120

from scifield.findings.seeding import (  # noqa: E402
    SPECIALTY_GROUPS,
    directed_seeding_network,
    seeding_by_group,
    seeding_score,
    specialty_of,
)
from scifield.repro import record_run  # noqa: E402

ARCHETYPES = DATA / "archetypes.parquet"
DUCKDB = DATA / "papers.duckdb"
# Shared provenance config fragment stamped onto every bonus sidecar.
BONUS_LAYER = "bonus_exploratory_non_gating"


def _load_parquet(path: Path, columns=None) -> pd.DataFrame | None:
    """Defensive parquet read — return None (with a message) if absent."""
    if not path.exists():
        print(f"MISSING artifact: {path} — cannot render this section.")
        return None
    return pd.read_parquet(path, columns=columns)


print("matplotlib:", matplotlib.__version__, "| backend:", matplotlib.get_backend())
print("duckdb    :", duckdb.__version__)
print("DATA       :", DATA)
print("FIGURES_DIR:", FIGURES_DIR)
print("specialty groups locked:", {k: len(v) for k, v in SPECIALTY_GROUPS.items()})

matplotlib: 3.10.9 | backend: Agg
duckdb    : 1.5.2
DATA       : /Users/samersalman/Desktop/SciField/data/v1
FIGURES_DIR: /Users/samersalman/Desktop/SciField/docs/figures
specialty groups locked: {'orthopedic': 10, 'general_surgery': 10}


## 2. Load archetypes + journal→slug→specialty mapping

`archetypes.parquet` carries the long display `journal` (e.g. *"The Journal of
arthroplasty"*), but `SPECIALTY_GROUPS` keys on the duckdb `journal_slug`
(`j_arthroplasty`) and the *short* display name. We therefore pull the distinct
`journal → journal_slug` map from the `papers` table and resolve `specialty_of(slug)`,
which covers all corpus journals (the two long names *"Archives of surgery"* and
*"JAMA surgery"* both map to slug `jama_surg` — Archives was renamed to JAMA Surgery,
so they are the same journal). We also build a short label per journal for readable
figures. Seeding is computed on **leaf topics only** (`topic_id != -1`): the `-1`
bucket is BERTopic noise, not a real research topic.

In [2]:
arch_all = _load_parquet(ARCHETYPES)
assert arch_all is not None, "archetypes.parquet missing — cannot proceed."
print(f"archetypes: {len(arch_all):,} rows | {arch_all['pmid'].nunique():,} unique pmid")
print(f"pmid dtype = {arch_all['pmid'].dtype} (int64; paper_institutions.pmid is STRING)")

# journal -> slug map (and a tidy short label) from the papers table.
con = duckdb.connect(str(DUCKDB), read_only=True)
slugmap = con.execute("SELECT DISTINCT journal, journal_slug FROM papers").df()
con.close()
journal_to_slug = dict(zip(slugmap["journal"], slugmap["journal_slug"], strict=False))

# Short, figure-friendly labels keyed by journal_slug.
SLUG_LABEL = {
    "spine": "Spine",
    "j_arthroplasty": "J Arthroplasty",
    "clin_orthop_relat_res": "Clin Orthop (CORR)",
    "j_bone_joint_surg_am": "J Bone Joint Surg",
    "arthroscopy": "Arthroscopy",
    "surgery": "Surgery",
    "ann_surg": "Ann Surg",
    "br_j_surg": "Br J Surg",
    "j_am_coll_surg": "J Am Coll Surg",
    "jama_surg": "JAMA Surg / Arch Surg",
}

# Leaf-topic working frame with slug, short label, and specialty attached.
arch = arch_all[arch_all["topic_id"] != -1].copy()
arch["journal_slug"] = arch["journal"].map(journal_to_slug)
arch["jlabel"] = arch["journal_slug"].map(SLUG_LABEL).fillna(arch["journal"])
arch["specialty"] = arch["journal_slug"].map(specialty_of)

n_unmapped_slug = int(arch["journal_slug"].isna().sum())
n_unmapped_spec = int(arch["specialty"].isna().sum())
print(f"\nleaf rows = {len(arch):,} across {arch['topic_id'].nunique()} topics")
print(f"rows with no journal_slug : {n_unmapped_slug}")
print(f"rows with no specialty     : {n_unmapped_spec} (expect 0 — all 10 journals map)")
print("\nspecialty coverage:")
print(arch["specialty"].value_counts(dropna=False).to_string())
assert n_unmapped_spec == 0, "some journals failed specialty resolution"

archetypes: 89,230 rows | 89,230 unique pmid
pmid dtype = int64 (int64; paper_institutions.pmid is STRING)



leaf rows = 67,821 across 149 topics
rows with no journal_slug : 0
rows with no specialty     : 0 (expect 0 — all 10 journals map)

specialty coverage:
specialty
orthopedic         40465
general_surgery    27356


## 3. UP-FRONT metadata-availability guard (required)

Before the institutional/geographic panel we **compute and print** affiliation /
country coverage. The guard decides whether the institutional panel RUNS and whether
the geographic panel carries a coverage caveat or DEGRADES to a documented limitation.
With the known corpus coverage both panels run; the degrade branch is coded
defensively but is not expected to trigger (no harvest is ever attempted — this layer
is strictly read-only).

In [3]:
con = duckdb.connect(str(DUCKDB), read_only=True)

n_papers_corpus = con.execute("SELECT COUNT(DISTINCT pmid) FROM papers").fetchone()[0]
n_pi_papers = con.execute("SELECT COUNT(DISTINCT pmid) FROM paper_institutions").fetchone()[0]
n_inst_total = con.execute("SELECT COUNT(*) FROM institutions").fetchone()[0]
n_inst_cc = con.execute(
    "SELECT COUNT(*) FROM institutions "
    "WHERE country_code IS NOT NULL AND TRIM(country_code) <> ''"
).fetchone()[0]

# Archetypes pmids (leaf) with >=1 affiliation — the population the panel actually uses.
con.execute(
    "CREATE TEMP TABLE arch_leaf_pmids AS "
    "SELECT DISTINCT CAST(pmid AS VARCHAR) AS pmid "
    "FROM read_parquet($p) WHERE topic_id <> -1",
    {"p": str(ARCHETYPES)},
)
n_arch_leaf = con.execute("SELECT COUNT(*) FROM arch_leaf_pmids").fetchone()[0]
n_arch_with_inst = con.execute(
    "SELECT COUNT(*) FROM arch_leaf_pmids a "
    "WHERE EXISTS (SELECT 1 FROM paper_institutions pi WHERE pi.pmid = a.pmid)"
).fetchone()[0]
con.close()

aff_corpus_frac = n_pi_papers / n_papers_corpus
aff_arch_frac = n_arch_with_inst / n_arch_leaf
country_inst_frac = n_inst_cc / n_inst_total

print("=== METADATA-AVAILABILITY GUARD (bonus, non-gating) ===")
print(f"corpus papers (papers table)        : {n_papers_corpus:,}")
print(f"papers with >=1 affiliation         : {n_pi_papers:,} ({aff_corpus_frac:.1%})")
print(f"archetypes-leaf papers              : {n_arch_leaf:,}")
print(f"  ...with >=1 affiliation           : {n_arch_with_inst:,} ({aff_arch_frac:.1%})")
print(f"institutions total                  : {n_inst_total:,}")
print(f"  ...with non-blank country_code    : {n_inst_cc:,} ({country_inst_frac:.1%})")

# Decision thresholds (documented). Institutional panel needs broad affiliation
# coverage; geographic panel runs but flags a caveat when full-table country
# coverage is partial. Neither branch triggers a harvest.
INST_PANEL_MIN = 0.50  # >=50% of archetypes-leaf papers must carry an affiliation
GEO_CAVEAT_BELOW = 0.90  # country present on < 90% of institutions -> caveat (not degrade)
GEO_DEGRADE_BELOW = 0.10  # country present on < 10% -> degrade to documented limitation

inst_panel_runs = aff_arch_frac >= INST_PANEL_MIN
geo_degrade = country_inst_frac < GEO_DEGRADE_BELOW
geo_caveat = (not geo_degrade) and (country_inst_frac < GEO_CAVEAT_BELOW)

print("\n-- guard verdict --")
print(f"institutional panel RUNS    : {inst_panel_runs}  (>= {INST_PANEL_MIN:.0%} affiliated)")
print(f"geographic panel DEGRADES   : {geo_degrade}  (would only fire < {GEO_DEGRADE_BELOW:.0%})")
print(f"geographic coverage CAVEAT  : {geo_caveat}  (country on {country_inst_frac:.0%} of insts)")

GEO_CAVEAT_TEXT = (
    f"Geographic seeding is exploratory: a country_code is present on only "
    f"{country_inst_frac:.0%} of institutions in the corpus institutions table "
    f"({n_inst_cc:,}/{n_inst_total:,}); blank-country rows are dropped (never imputed), "
    f"so absent countries cannot masquerade as seeders. Country ranks are therefore "
    f"over the affiliated, country-tagged subset only."
)
print("\ncoverage caveat wording (for downstream):")
print(" ", GEO_CAVEAT_TEXT)
assert inst_panel_runs, "affiliation coverage unexpectedly thin — see guard output"

=== METADATA-AVAILABILITY GUARD (bonus, non-gating) ===
corpus papers (papers table)        : 121,908
papers with >=1 affiliation         : 113,413 (93.0%)
archetypes-leaf papers              : 67,821
  ...with >=1 affiliation           : 65,470 (96.5%)
institutions total                  : 29,159
  ...with non-blank country_code    : 16,258 (55.8%)

-- guard verdict --
institutional panel RUNS    : True  (>= 50% affiliated)
geographic panel DEGRADES   : False  (would only fire < 10%)
geographic coverage CAVEAT  : True  (country on 56% of insts)

coverage caveat wording (for downstream):
  Geographic seeding is exploratory: a country_code is present on only 56% of institutions in the corpus institutions table (16,258/29,159); blank-country rows are dropped (never imputed), so absent countries cannot masquerade as seeders. Country ranks are therefore over the affiliated, country-tagged subset only.


## 4. Sub-analysis 1 — cross-journal seeding (exploratory / non-gating)

`seeding_score` gives each journal a mean normalized-lead score across the topics it
participates in (1.0 = always earliest, 0.0 = always latest). We score on the canonical
**`journal_slug`** (10 corpus journals) rather than the raw display name so that
*"Archives of surgery"* and *"JAMA surgery"* — the same journal before/after its 2013
rename — collapse into one `jama_surg` node instead of two artefactual entities.
`directed_seeding_network` returns a pandas edge list `[src, dst, n_precedes, n_shared,
weight]`; a high-weight edge `src → dst` means `src` tends to publish before `dst`. We
draw the directed network with **matplotlib only** (no networkx), thresholding on
`weight >= 0.6`.

In [4]:
# Score on canonical journal_slug (10 journals) so JAMA/Archives merge.
seed_journal = seeding_score(arch[["topic_id", "journal_slug", "year"]], entity_col="journal_slug")
seed_journal["jlabel"] = (
    seed_journal["journal_slug"].map(SLUG_LABEL).fillna(seed_journal["journal_slug"])
)
seed_journal["specialty"] = seed_journal["journal_slug"].map(specialty_of)
print("=== per-journal seeding score (higher = seeds / publishes first) ===")
print(
    seed_journal[["jlabel", "specialty", "seeding_score", "n_topics"]]
    .round(3)
    .to_string(index=False)
)

net = directed_seeding_network(
    arch[["topic_id", "journal_slug", "year"]], entity_col="journal_slug"
)
net["src_label"] = net["src"].map(SLUG_LABEL).fillna(net["src"])
net["dst_label"] = net["dst"].map(SLUG_LABEL).fillna(net["dst"])
WEIGHT_THRESHOLD = 0.6
strong = net[net["weight"] >= WEIGHT_THRESHOLD].copy()
print(f"\nnetwork: {len(net)} directed edges | {len(strong)} with weight >= {WEIGHT_THRESHOLD}")

top_seed = seed_journal.iloc[0]
bot_seed = seed_journal.iloc[-1]
print(
    f"\nSEEDS (publishes first): {top_seed['jlabel']} "
    f"(score {top_seed['seeding_score']:.3f}, {int(top_seed['n_topics'])} topics)"
)
print(
    f"FOLLOWS (publishes late): {bot_seed['jlabel']} "
    f"(score {bot_seed['seeding_score']:.3f}, {int(bot_seed['n_topics'])} topics)"
)

=== per-journal seeding score (higher = seeds / publishes first) ===
               jlabel       specialty  seeding_score  n_topics
            Br J Surg general_surgery          0.739        89
    J Bone Joint Surg      orthopedic          0.702       101
   Clin Orthop (CORR)      orthopedic          0.692       115
              Surgery general_surgery          0.679        94
       J Am Coll Surg general_surgery          0.659        95
             Ann Surg general_surgery          0.632        97
JAMA Surg / Arch Surg general_surgery          0.627       102
                Spine      orthopedic          0.538        82
       J Arthroplasty      orthopedic          0.480        68
          Arthroscopy      orthopedic          0.463        66

network: 90 directed edges | 6 with weight >= 0.6

SEEDS (publishes first): Br J Surg (score 0.739, 89 topics)
FOLLOWS (publishes late): Arthroscopy (score 0.463, 66 topics)


In [5]:
# --- Figure 1: bonus_seeding_network.png (matplotlib, no networkx) ---
# Layout: circular by seeding rank (strongest seeder at top, clockwise). Node colour
# = specialty, node size = n_topics, edge alpha/width = weight; arrows src -> dst.
order = seed_journal.sort_values("seeding_score", ascending=False).reset_index(drop=True)
n_nodes = len(order)
angles = np.linspace(np.pi / 2, np.pi / 2 - 2 * np.pi, n_nodes, endpoint=False)
pos = {}  # keyed by journal_slug to match the network's src/dst
for ang, jrow in zip(angles, order.itertuples(), strict=False):
    pos[jrow.journal_slug] = (np.cos(ang), np.sin(ang))

spec_color = {"orthopedic": "#4285f4", "general_surgery": "#ea4335", None: "#9aa0a6"}
score_lookup = dict(zip(order["journal_slug"], order["seeding_score"], strict=False))
ntopic_lookup = dict(zip(order["journal_slug"], order["n_topics"], strict=False))

fig, (axn, axb) = plt.subplots(1, 2, figsize=(15, 7.2), gridspec_kw={"width_ratios": [1.25, 1]})

# Edges first (under nodes).
if len(strong):
    wmin, wmax = strong["weight"].min(), strong["weight"].max()
    for e in strong.itertuples():
        x0, y0 = pos[e.src]
        x1, y1 = pos[e.dst]
        frac = 0.0 if wmax == wmin else (e.weight - wmin) / (wmax - wmin)
        axn.annotate(
            "",
            xy=(x1, y1),
            xytext=(x0, y0),
            arrowprops=dict(
                arrowstyle="-|>",
                color="#555555",
                alpha=0.25 + 0.55 * frac,
                lw=0.6 + 2.4 * frac,
                shrinkA=14,
                shrinkB=16,
                connectionstyle="arc3,rad=0.08",
            ),
        )

for jrow in order.itertuples():
    slug = jrow.journal_slug
    x, y = pos[slug]
    spec = specialty_of(slug)
    size = 280 + 12 * float(ntopic_lookup[slug])
    axn.scatter(x, y, s=size, c=spec_color.get(spec), edgecolors="white", linewidths=1.5, zorder=3)
    label = SLUG_LABEL.get(slug, slug)
    ha = "left" if x > 0.05 else ("right" if x < -0.05 else "center")
    axn.text(
        x * 1.18,
        y * 1.18,
        f"{label}\n{score_lookup[slug]:.2f}",
        ha=ha,
        va="center",
        fontsize=8,
    )

axn.set_xlim(-1.7, 1.7)
axn.set_ylim(-1.45, 1.45)
axn.axis("off")
axn.set_title(
    f"(a) Directed cross-journal seeding network\n"
    f"edge src->dst: src publishes first (weight >= {WEIGHT_THRESHOLD}); "
    f"node size ~ #topics, label = seeding score",
    fontsize=9,
)
handles = [
    plt.Line2D([0], [0], marker="o", ls="", mfc="#4285f4", mec="white", ms=11, label="orthopedic"),
    plt.Line2D(
        [0], [0], marker="o", ls="", mfc="#ea4335", mec="white", ms=11, label="general_surgery"
    ),
]
axn.legend(handles=handles, fontsize=8, loc="lower right", frameon=False)

# Companion bar: seeding score ranked, coloured by specialty.
bar_colors = [spec_color.get(specialty_of(s)) for s in order["journal_slug"]]
ypos = np.arange(n_nodes)[::-1]
axb.barh(ypos, order["seeding_score"].to_numpy(), color=bar_colors)
axb.set_yticks(ypos)
axb.set_yticklabels([SLUG_LABEL.get(s, s) for s in order["journal_slug"]], fontsize=8)
axb.axvline(0.5, ls=":", c="#888888", lw=1)
axb.set_xlim(0, 1)
axb.set_xlabel("seeding score (mean normalized lead)")
axb.set_title("(b) Journal seeding ranking\n1.0 = always earliest, 0.0 = always latest", fontsize=9)

fig.suptitle(
    "BONUS (exploratory / non-gating): cross-journal seeding — who publishes first within a topic",
    fontsize=12,
)
fig.tight_layout(rect=[0, 0, 1, 0.95])
FIG1 = FIGURES_DIR / "bonus_seeding_network.png"
fig.savefig(FIG1, dpi=DPI)
plt.close(fig)
sz1 = FIG1.stat().st_size
print(f"wrote {FIG1.name} | {sz1 / 1024:.1f} KB")
assert sz1 < 1_000_000, f"FIG1 too large: {sz1} bytes"

wrote bonus_seeding_network.png | 104.5 KB


In [6]:
sidecar1 = record_run(
    artifact_path=FIG1,
    inputs={"archetypes": ARCHETYPES},
    config={
        "figure": "bonus_seeding_network",
        "layer": BONUS_LAYER,
        "session": "V1-S15",
        "dpi": DPI,
        "weight_threshold": WEIGHT_THRESHOLD,
        "entity_col": "journal_slug",
        "n_leaf_topics": int(arch["topic_id"].nunique()),
        "n_journals": int(len(seed_journal)),
        "n_edges_total": int(len(net)),
        "n_edges_thresholded": int(len(strong)),
        "top_seeder": top_seed["jlabel"],
        "top_seeder_score": float(top_seed["seeding_score"]),
        "top_follower": bot_seed["jlabel"],
        "top_follower_score": float(bot_seed["seeding_score"]),
    },
)
print("recorded sidecar:", sidecar1.name)

recorded sidecar: bonus_seeding_network.png.run.json


## 5. Sub-analysis 2 — per-specialty contrast (exploratory / non-gating)

Orthopedic vs general_surgery (the locked 5-vs-5 anatomical split). We contrast
(a) the **F2 archetype mix** (`arch_mean_cd5` category shares) between specialties and
(b) **seeding patterns** by specialty via `seeding_by_group(group_col="specialty")`.
The F1 epistemic cascade is **NULL overall** (read from `f1_cascade_verdict.json`), so
any per-specialty cascade reading would be exploratory only — we surface the F2 mix and
seeding, not a cascade claim.

In [7]:
# F1 verdict context (read-only) — to caveat the per-specialty narrative.
f1_verdict_path = DATA / "f1_cascade_verdict.json"
f1_verdict = "unknown"
if f1_verdict_path.exists():
    f1_verdict = json.loads(f1_verdict_path.read_text()).get("f1_verdict", "unknown")
print(
    f"F1 epistemic cascade (overall corpus verdict) = {f1_verdict} "
    f"-> per-specialty cascade notes would be EXPLORATORY only"
)

ARCH_ORDER = ["disruptive-novel", "conventional-disruptive", "novel-consolidating", "incremental"]
mix_src = arch.dropna(subset=["arch_mean_cd5"]).copy()
mix = pd.crosstab(mix_src["specialty"], mix_src["arch_mean_cd5"], normalize="index").reindex(
    columns=ARCH_ORDER
)
print("\n=== (a) F2 arch_mean_cd5 MIX by specialty (row-normalized shares) ===")
print(mix.round(3).to_string())

seed_spec = seeding_by_group(arch[["topic_id", "specialty", "year"]], group_col="specialty")
print("\n=== (b) seeding_by_group(specialty) ===")
print(seed_spec.round(3).to_string(index=False))

# Biggest archetype-share gap between the two specialties (descriptive).
gap = (mix.loc["orthopedic"] - mix.loc["general_surgery"]).sort_values()
print("\narchetype-share gap (orthopedic - general_surgery):")
print(gap.round(3).to_string())

F1 epistemic cascade (overall corpus verdict) = NULL -> per-specialty cascade notes would be EXPLORATORY only



=== (a) F2 arch_mean_cd5 MIX by specialty (row-normalized shares) ===
arch_mean_cd5    disruptive-novel  conventional-disruptive  novel-consolidating  incremental
specialty                                                                                   
general_surgery             0.256                    0.278                0.175        0.290
orthopedic                  0.155                    0.336                0.131        0.379

=== (b) seeding_by_group(specialty) ===
      specialty  seeding_score  n_topics
     orthopedic          0.740       123
general_surgery          0.675       117

archetype-share gap (orthopedic - general_surgery):
arch_mean_cd5
disruptive-novel          -0.102
novel-consolidating       -0.044
conventional-disruptive    0.057
incremental                0.089


In [8]:
# --- Figure 2: bonus_specialty_contrast.png ---
fig, (axm, axs) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1.5, 1]})

# (a) grouped archetype-mix bars.
spec_list = ["orthopedic", "general_surgery"]
arch_palette = {
    "disruptive-novel": "#ea4335",
    "conventional-disruptive": "#fbbc04",
    "novel-consolidating": "#34a853",
    "incremental": "#4285f4",
}
xpos = np.arange(len(ARCH_ORDER))
bw = 0.38
for i, sp in enumerate(spec_list):
    vals = mix.loc[sp, ARCH_ORDER].to_numpy()
    bars = axm.bar(
        xpos + (i - 0.5) * bw,
        vals,
        width=bw,
        label=sp,
        color="#4285f4" if sp == "orthopedic" else "#ea4335",
        edgecolor="white",
    )
    axm.bar_label(bars, fmt="%.2f", fontsize=7, padding=1)
axm.set_xticks(xpos)
axm.set_xticklabels([a.replace("-", "-\n") for a in ARCH_ORDER], fontsize=8)
axm.set_ylabel("share of papers within specialty")
axm.set_ylim(0, max(mix.to_numpy().max() * 1.18, 0.45))
axm.set_title(
    "(a) F2 archetype mix by specialty\n(arch_mean_cd5 category shares; descriptive)", fontsize=9
)
axm.legend(fontsize=8)

# (b) seeding score by specialty.
sp_order = seed_spec.sort_values("seeding_score", ascending=False)
bcols = ["#4285f4" if s == "orthopedic" else "#ea4335" for s in sp_order["specialty"]]
bars2 = axs.bar(sp_order["specialty"], sp_order["seeding_score"], color=bcols, edgecolor="white")
axs.bar_label(
    bars2,
    labels=[
        f"{v:.3f}\n(n_top={int(n)})"
        for v, n in zip(sp_order["seeding_score"], sp_order["n_topics"], strict=False)
    ],
    fontsize=8,
)
axs.axhline(0.5, ls=":", c="#888888", lw=1)
axs.set_ylim(0, 1)
axs.set_ylabel("seeding score (mean normalized lead)")
axs.set_title("(b) Seeding by specialty\nhigher = publishes first within shared topics", fontsize=9)
axs.tick_params(axis="x", rotation=10)

fig.suptitle(
    "BONUS (exploratory / non-gating): orthopedic vs general_surgery — "
    f"archetype mix & seeding  |  F1 cascade overall = {f1_verdict}",
    fontsize=11,
)
fig.tight_layout(rect=[0, 0, 1, 0.94])
FIG2 = FIGURES_DIR / "bonus_specialty_contrast.png"
fig.savefig(FIG2, dpi=DPI)
plt.close(fig)
sz2 = FIG2.stat().st_size
print(f"wrote {FIG2.name} | {sz2 / 1024:.1f} KB")
assert sz2 < 1_000_000, f"FIG2 too large: {sz2} bytes"

wrote bonus_specialty_contrast.png | 79.0 KB


In [9]:
sidecar2 = record_run(
    artifact_path=FIG2,
    inputs={"archetypes": ARCHETYPES, "f1_cascade_verdict": f1_verdict_path},
    config={
        "figure": "bonus_specialty_contrast",
        "layer": BONUS_LAYER,
        "session": "V1-S15",
        "dpi": DPI,
        "specialties": spec_list,
        "arch_categories": ARCH_ORDER,
        "f1_cascade_overall_verdict": f1_verdict,
        "seeding_ortho": float(seed_spec.set_index("specialty").loc["orthopedic", "seeding_score"]),
        "seeding_gen_surg": float(
            seed_spec.set_index("specialty").loc["general_surgery", "seeding_score"]
        ),
        "largest_arch_gap_category": str(gap.abs().idxmax()),
        "largest_arch_gap_value": float(gap.loc[gap.abs().idxmax()]),
    },
)
print("recorded sidecar:", sidecar2.name)

recorded sidecar: bonus_specialty_contrast.png.run.json


## 6. Sub-analysis 3 — institutional / geographic seeding (exploratory / non-gating)

Build one row per **(paper, institution)** by joining `paper_institutions` →
archetypes on `pmid` (archetypes `pmid` is int64, `paper_institutions.pmid` is STRING —
we CAST), carrying `topic_id, year, institution_canonical_id`, then LEFT-join
`institutions` for `country_code`. `seeding_by_group(group_col="institution_canonical_id")`
ranks seeding institutions; `seeding_by_group(group_col="country_code")` ranks countries
(blank countries are **dropped** by the module — the coverage caveat from the guard).
Institutions are shown with a stability floor on `n_topics` so the ranking reflects
well-established centres rather than niche one-topic entities.

In [10]:
con = duckdb.connect(str(DUCKDB), read_only=True)
merged = con.execute(
    """
    SELECT a.topic_id,
           a.year,
           pi.institution_canonical_id,
           inst.country_code,
           inst.display_name AS inst_name
    FROM read_parquet($p) a
    JOIN paper_institutions pi ON CAST(a.pmid AS VARCHAR) = pi.pmid
    LEFT JOIN institutions inst
           ON pi.institution_canonical_id = inst.institution_canonical_id
    WHERE a.topic_id <> -1
    """,
    {"p": str(ARCHETYPES)},
).df()
con.close()

merged_country_frac = (merged["country_code"].fillna("").str.strip() != "").mean()
print(f"merged (paper x institution) rows : {len(merged):,}")
print(f"unique institutions               : {merged['institution_canonical_id'].nunique():,}")
print(
    f"rows with non-blank country       : {merged_country_frac:.1%} "
    f"(affiliated subset is better-covered than the full institutions table)"
)

# Institution seeding, with a stability floor so niche 1-topic entities don't dominate.
INST_MIN_TOPICS = 80
seed_inst = seeding_by_group(
    merged[["topic_id", "institution_canonical_id", "year"]],
    group_col="institution_canonical_id",
)
id2name = dict(zip(merged["institution_canonical_id"], merged["inst_name"], strict=False))
seed_inst["name"] = seed_inst["institution_canonical_id"].map(id2name)
inst_stable = seed_inst[seed_inst["n_topics"] >= INST_MIN_TOPICS].copy()
print(
    f"\n=== top seeding institutions (n_topics >= {INST_MIN_TOPICS}; "
    f"{len(inst_stable)} qualify) ==="
)
print(inst_stable.head(12)[["name", "seeding_score", "n_topics"]].round(3).to_string(index=False))

# Country seeding (blank countries dropped by the module = the coverage caveat).
COUNTRY_MIN_TOPICS = 30
seed_country = seeding_by_group(
    merged[["topic_id", "country_code", "year"]], group_col="country_code"
)
country_stable = seed_country[seed_country["n_topics"] >= COUNTRY_MIN_TOPICS].copy()
print(
    f"\n=== top seeding countries (n_topics >= {COUNTRY_MIN_TOPICS}; "
    f"{len(country_stable)} qualify of {len(seed_country)} total) ==="
)
print(country_stable.head(12).round(3).to_string(index=False))

merged (paper x institution) rows : 585,958


unique institutions               : 21,803
rows with non-blank country       : 94.4% (affiliated subset is better-covered than the full institutions table)



=== top seeding institutions (n_topics >= 80; 65 qualify) ===
                                   name  seeding_score  n_topics
                            Mayo Clinic          0.822       125
           Hospital for Special Surgery          0.782        81
University of California, San Francisco          0.780       116
                     Harvard University          0.771       136
               Johns Hopkins University          0.757       128
         Massachusetts General Hospital          0.755       129
     Washington University in St. Louis          0.746       119
                    Duke Medical Center          0.742       110
           Brigham and Women's Hospital          0.739       122
         Rush University Medical Center          0.734        95
               University of Washington          0.726       109
                  University of Toronto          0.724       115

=== top seeding countries (n_topics >= 30; 54 qualify of 164 total) ===
country_code  seedi

In [11]:
# --- Figure 3: bonus_geographic.png (top seeding institutions + countries) ---
fig, (axi, axc) = plt.subplots(1, 2, figsize=(14.5, 6.4))

# (a) top seeding institutions.
top_inst = inst_stable.head(10).iloc[::-1]
yi = np.arange(len(top_inst))
axi.barh(yi, top_inst["seeding_score"].to_numpy(), color="#34a853", edgecolor="white")
axi.set_yticks(yi)
axi.set_yticklabels(
    [
        (str(n)[:34] + "...") if isinstance(n, str) and len(str(n)) > 34 else str(n)
        for n in top_inst["name"]
    ],
    fontsize=8,
)
for y, (s, nt) in enumerate(zip(top_inst["seeding_score"], top_inst["n_topics"], strict=False)):
    axi.text(s + 0.005, y, f"{s:.2f} (n={int(nt)})", va="center", fontsize=7)
axi.axvline(0.5, ls=":", c="#888888", lw=1)
axi.set_xlim(0, 1.08)
axi.set_xlabel("seeding score (mean normalized lead)")
axi.set_title(
    f"(a) Top seeding institutions (n_topics >= {INST_MIN_TOPICS})\n"
    "higher = consistently publishes first within a topic",
    fontsize=9,
)

# (b) top seeding countries.
top_country = country_stable.head(12).iloc[::-1]
yc = np.arange(len(top_country))
axc.barh(yc, top_country["seeding_score"].to_numpy(), color="#4285f4", edgecolor="white")
axc.set_yticks(yc)
axc.set_yticklabels(top_country["country_code"].tolist(), fontsize=8)
for y, (s, nt) in enumerate(
    zip(top_country["seeding_score"], top_country["n_topics"], strict=False)
):
    axc.text(s + 0.005, y, f"{s:.2f} (n={int(nt)})", va="center", fontsize=7)
axc.axvline(0.5, ls=":", c="#888888", lw=1)
axc.set_xlim(0, 1.08)
axc.set_xlabel("seeding score (mean normalized lead)")
axc.set_title(
    f"(b) Top seeding countries (n_topics >= {COUNTRY_MIN_TOPICS})\n"
    f"country present on {country_inst_frac:.0%} of insts (blanks dropped, caveat)",
    fontsize=9,
)

fig.suptitle(
    "BONUS (exploratory / non-gating): institutional & geographic seeding "
    "(ISO country codes; not imputed)",
    fontsize=11,
)
fig.tight_layout(rect=[0, 0, 1, 0.94])
FIG3 = FIGURES_DIR / "bonus_geographic.png"
fig.savefig(FIG3, dpi=DPI)
plt.close(fig)
sz3 = FIG3.stat().st_size
print(f"wrote {FIG3.name} | {sz3 / 1024:.1f} KB")
assert sz3 < 1_000_000, f"FIG3 too large: {sz3} bytes"

wrote bonus_geographic.png | 96.6 KB


In [12]:
sidecar3 = record_run(
    artifact_path=FIG3,
    inputs={"archetypes": ARCHETYPES, "papers_duckdb": DUCKDB},
    config={
        "figure": "bonus_geographic",
        "layer": BONUS_LAYER,
        "session": "V1-S15",
        "dpi": DPI,
        "inst_min_topics": INST_MIN_TOPICS,
        "country_min_topics": COUNTRY_MIN_TOPICS,
        "n_paper_institution_rows": int(len(merged)),
        "n_unique_institutions": int(merged["institution_canonical_id"].nunique()),
        "country_coverage_full_table": float(country_inst_frac),
        "country_coverage_affiliated_rows": float(merged_country_frac),
        "affiliation_coverage_corpus": float(aff_corpus_frac),
        "affiliation_coverage_archetypes_leaf": float(aff_arch_frac),
        "geographic_caveat": GEO_CAVEAT_TEXT,
        "top_institution": str(inst_stable.iloc[0]["name"]),
        "top_institution_score": float(inst_stable.iloc[0]["seeding_score"]),
        "top_country": str(country_stable.iloc[0]["country_code"]),
        "top_country_score": float(country_stable.iloc[0]["seeding_score"]),
    },
)
print("recorded sidecar:", sidecar3.name)

recorded sidecar: bonus_geographic.png.run.json


## 7. Verify bonus artifacts

Confirm all three figures + sidecars exist and each figure is < 1 MB. This layer is
**exploratory / non-gating** — it does **not** contribute to the Gate G5 count.

In [13]:
bonus_figs = [FIG1, FIG2, FIG3]
print("=== BONUS artifacts (exploratory / non-gating) ===")
for f in bonus_figs:
    sc = f.with_suffix(f.suffix + ".run.json")
    ok = f.exists() and sc.exists()
    kb = f.stat().st_size / 1024 if f.exists() else float("nan")
    print(f"  [{'ok' if ok else 'XX'}] {f.name:<30} {kb:6.1f} KB  + sidecar={sc.exists()}")
    assert f.exists(), f"missing figure {f}"
    assert sc.exists(), f"missing sidecar for {f}"
    assert f.stat().st_size < 1_000_000, f"{f} exceeds 1 MB"

print(f"\n{len(bonus_figs)} bonus figures written (<= 3), each < 1 MB, each with a sidecar.")

_seed_by_spec = seed_spec.set_index("specialty")["seeding_score"]
_ortho = float(_seed_by_spec.loc["orthopedic"])
_gen = float(_seed_by_spec.loc["general_surgery"])
_top_inst_name = str(inst_stable.iloc[0]["name"])
_top_inst_score = float(inst_stable.iloc[0]["seeding_score"])
_top_countries = ", ".join(country_stable.head(5)["country_code"].tolist())

print("\nSEEDING SUMMARY (exploratory):")
print(
    f"  journals  — seeds: {top_seed['jlabel']} ({top_seed['seeding_score']:.2f}) | "
    f"follows: {bot_seed['jlabel']} ({bot_seed['seeding_score']:.2f})"
)
print(f"  specialty — ortho {_ortho:.3f} vs gen_surg {_gen:.3f}")
print(f"  institution — top seeder: {_top_inst_name} ({_top_inst_score:.2f})")
print(f"  country — top seeders: {_top_countries}")
print("\nReminder: NON-GATING bonus layer — excluded from the Gate G5 finding count.")

=== BONUS artifacts (exploratory / non-gating) ===
  [ok] bonus_seeding_network.png       104.5 KB  + sidecar=True
  [ok] bonus_specialty_contrast.png     79.0 KB  + sidecar=True
  [ok] bonus_geographic.png             96.6 KB  + sidecar=True

3 bonus figures written (<= 3), each < 1 MB, each with a sidecar.

SEEDING SUMMARY (exploratory):
  journals  — seeds: Br J Surg (0.74) | follows: Arthroscopy (0.46)
  specialty — ortho 0.740 vs gen_surg 0.675
  institution — top seeder: Mayo Clinic (0.82)
  country — top seeders: US, GB, JP, CA, DE

Reminder: NON-GATING bonus layer — excluded from the Gate G5 finding count.
